# **Uplift Modelling of a Marketing Campaign.**

## **Load Data.**

In [1]:
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/MarketingUpliftModelling/"

data_df = pd.read_csv(DATASETS_PATH + "Kevin_Hillstrom_MineThatData_E-MailAnalytics.csv")

print("Data: ")
print(data_df)

Data: 
       recency history_segment  history  mens  womens   zip_code  newbie  \
0           10  2) $100 - $200   142.44     1       0  Surburban       0   
1            6  3) $200 - $350   329.08     1       1      Rural       1   
2            7  2) $100 - $200   180.65     0       1  Surburban       1   
3            9  5) $500 - $750   675.83     1       0      Rural       1   
4            2    1) $0 - $100    45.34     1       0      Urban       0   
...        ...             ...      ...   ...     ...        ...     ...   
63995       10  2) $100 - $200   105.54     1       0      Urban       0   
63996        5    1) $0 - $100    38.91     0       1      Urban       1   
63997        6    1) $0 - $100    29.99     1       0      Urban       1   
63998        1  5) $500 - $750   552.94     1       0  Surburban       1   
63999        1  4) $350 - $500   472.82     0       1  Surburban       0   

            channel        segment  visit  conversion  spend  
0             Pho

## **Encode categorical string variables.**

In [2]:
data_df["zip_code"] = data_df["zip_code"].astype("category")
data_df["zip_code_cat"] = data_df["zip_code"].cat.codes

data_df["channel"] = data_df["channel"].astype("category")
data_df["channel_cat"] = data_df["channel"].cat.codes


## **Encode treatment variable and create features and target frames.**

In [3]:
data_df["treatment"] = (data_df["segment"] != "No E-Mail").astype(int)

y = data_df["conversion"]
treatment = data_df["treatment"]

X = data_df[["recency", "history", "newbie", "mens", "womens", "zip_code_cat", "channel_cat"]]


## **Train an ensamble uplift model consisting of lightGBM.**

Uplift modeling requires estimating the difference between two conditional probabilities (treatment vs control). This makes the task inherently noisy and high-variance, especially in marketing datasets where treatment effects are small. Initial experiments with single uplift models (S-learner and tree-based approaches) showed unstable performance across different train-test splits, with Qini scores varying significantly. This indicates high variance in treatment effect estimation. To improve robustness, an ensemble approach was used. By averaging predictions from multiple models trained with different random seeds, variance in uplift estimation is reduced, leading to more stable and reliable customer ranking.

In [4]:
from lightgbm import LGBMClassifier
import numpy as np
from sklearn.model_selection import train_test_split
from sklift.metrics import qini_auc_score

X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    X, y, treatment, test_size=0.3, random_state=42, stratify=treatment
)

n_models = 20
models = []

for seed in range(n_models):

    model = LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=5,
        max_depth=4,
        subsample=0.7,
        colsample_bytree=0.7,
        random_state=seed
    )

    X_train_s = X_train.copy()
    X_train_s["treatment"] = t_train

    model.fit(X_train_s, y_train)
    models.append(model)

def predict_uplift(model, X):
    X_t = X.copy()
    X_c = X.copy()

    X_t["treatment"] = 1
    X_c["treatment"] = 0

    return (
        model.predict_proba(X_t)[:, 1]
        - model.predict_proba(X_c)[:, 1]
    )

uplift_matrix = np.array([
    predict_uplift(m, X_test) for m in models
])

uplift_test = uplift_matrix.mean(axis=0)

from sklift.metrics import qini_auc_score

qini = qini_auc_score(
    y_true=y_test,
    uplift=uplift_test,
    treatment=t_test
)
print()
print()
print("Ensemble Qini:", qini)


[LightGBM] [Info] Number of positive: 397, number of negative: 44403
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000419 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 282
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.008862 -> initscore=-4.717126
[LightGBM] [Info] Start training from score -4.717126
[LightGBM] [Info] Number of positive: 397, number of negative: 44403
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000510 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 282
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 8
[LightGBM] [Info] [binary:

In [5]:
uplift_full = np.mean([
    predict_uplift(m, X) for m in models
], axis=0)

data_df["uplift_score"] = uplift_full
print(data_df)

       recency history_segment  history  mens  womens   zip_code  newbie  \
0           10  2) $100 - $200   142.44     1       0  Surburban       0   
1            6  3) $200 - $350   329.08     1       1      Rural       1   
2            7  2) $100 - $200   180.65     0       1  Surburban       1   
3            9  5) $500 - $750   675.83     1       0      Rural       1   
4            2    1) $0 - $100    45.34     1       0      Urban       0   
...        ...             ...      ...   ...     ...        ...     ...   
63995       10  2) $100 - $200   105.54     1       0      Urban       0   
63996        5    1) $0 - $100    38.91     0       1      Urban       1   
63997        6    1) $0 - $100    29.99     1       0      Urban       1   
63998        1  5) $500 - $750   552.94     1       0  Surburban       1   
63999        1  4) $350 - $500   472.82     0       1  Surburban       0   

            channel        segment  visit  conversion  spend  zip_code_cat  \
0        

The ansamble **Qini-Score** in this case is around 0.077, which is a good performance for an uplift model. So no further model fine-tuning is needed.

## **Compare the trained ensamble uplift models to a random selection.**

In [6]:
k = 0.1  # top 10%
n_target = int(len(data_df) * k)

data_df_sorted = data_df.sort_values("uplift_score", ascending=False)

uplift_target = data_df_sorted.head(n_target)

random_target = data_df.sample(n=n_target, random_state=42)

uplift_rate = uplift_target["conversion"].mean()
random_rate = random_target["conversion"].mean()

absolute_lift = uplift_rate - random_rate
relative_lift = uplift_rate / random_rate if random_rate > 0 else np.nan

print("Uplift policy response rate:", uplift_rate)
print("Random policy response rate:", random_rate)
print("Absolute uplift gain:", absolute_lift)
print("Relative lift:", relative_lift)

Uplift policy response rate: 0.020625
Random policy response rate: 0.00890625
Absolute uplift gain: 0.011718750000000002
Relative lift: 2.3157894736842106


Absolute lift of around 1.17% means that for every 100 customers random targeting delivers 0.9 conversions and the uplift model 2.1, which is aditional 1.2 conversions. So the uplift model targeting is 2.3 times stronger than random.

## **Define the customers into 'persuadables', 'sure thing' and 'do not disturb' groups.**

In [7]:
X_t = X.copy()
X_c = X.copy()

X_t["treatment"] = 1
X_c["treatment"] = 0

p_t = np.mean([m.predict_proba(X_t)[:, 1] for m in models], axis=0)
p_c = np.mean([m.predict_proba(X_c)[:, 1] for m in models], axis=0)

data_df["p_t"] = p_t
data_df["p_c"] = p_c

uplift_thr = data_df["uplift_score"].quantile(0.8)
neg_thr = data_df["uplift_score"].quantile(0.2)
pc_thr = data_df["p_c"].median()

def segment_customer(row):
    if row["uplift_score"] > uplift_thr:
        return "Persuadable"

    elif row["uplift_score"] < neg_thr:
        return "Do Not Disturb"

    elif row["p_c"] > pc_thr:
        return "Sure Thing"

    else:
        return "Lost Cause"

data_df["uplift_segment"] = data_df.apply(segment_customer, axis=1)
print(data_df[data_df["uplift_segment"] == "Persuadable"])

       recency   history_segment  history  mens  womens   zip_code  newbie  \
3            9    5) $500 - $750   675.83     1       0      Rural       1   
8            9    5) $500 - $750   675.07     1       1      Rural       1   
9           10      1) $0 - $100    32.84     0       1      Urban       1   
12           5    5) $500 - $750   642.90     0       1  Surburban       1   
19           5  6) $750 - $1,000   828.42     1       0  Surburban       1   
...        ...               ...      ...   ...     ...        ...     ...   
63976        1    5) $500 - $750   710.72     1       1      Urban       1   
63979       10    2) $100 - $200   168.21     0       1  Surburban       0   
63986        9      1) $0 - $100    35.26     0       1      Urban       1   
63992        1    5) $500 - $750   519.69     1       1      Urban       1   
63998        1    5) $500 - $750   552.94     1       0  Surburban       1   

            channel        segment  visit  conversion  spend  z

## **Create pursuadables profile considering features of the customers.**

In [8]:
persuadables = data_df[data_df["uplift_segment"] == "Persuadable"]
overall = data_df.copy()

features_num = ["recency", "history"]

print("Persuadables:")
print(persuadables[features_num].mean())

print("\nOverall:")
print(overall[features_num].mean())

features_cat = ["channel", "zip_code", "newbie", "mens", "womens"]

for col in features_cat:
    print(f"\n{col} distribution (Persuadables):")
    print(persuadables[col].value_counts(normalize=True))
    
    print(f"\n{col} distribution (Overall):")
    print(overall[col].value_counts(normalize=True))

def compare_distribution(df1, df2, col):
    dist1 = df1[col].value_counts(normalize=True)
    dist2 = df2[col].value_counts(normalize=True)
    
    return (dist1 - dist2).sort_values(ascending=False)

compare_distribution(persuadables, overall, "channel")

importance = models[0].feature_importances_
feat_names = X.columns.tolist() + ["treatment"]

feature_importance = pd.Series(importance, index=feat_names).sort_values(ascending=False)
print()
print("Feature Importance: ")
print(str(feature_importance))

Persuadables:
recency      5.335000
history    535.673591
dtype: float64

Overall:
recency      5.763734
history    242.085656
dtype: float64

channel distribution (Persuadables):
channel
Web             0.459766
Multichannel    0.321172
Phone           0.219062
Name: proportion, dtype: float64

channel distribution (Overall):
channel
Web             0.440891
Phone           0.437828
Multichannel    0.121281
Name: proportion, dtype: float64

zip_code distribution (Persuadables):
zip_code
Surburban    0.426172
Urban        0.385781
Rural        0.188047
Name: proportion, dtype: float64

zip_code distribution (Overall):
zip_code
Surburban    0.449625
Urban        0.400953
Rural        0.149422
Name: proportion, dtype: float64

newbie distribution (Persuadables):
newbie
1    0.776484
0    0.223516
Name: proportion, dtype: float64

newbie distribution (Overall):
newbie
1    0.50225
0    0.49775
Name: proportion, dtype: float64

mens distribution (Persuadables):
mens
1    0.5425
0    0.4575

## **Core profile of the pursuadables.**

Comparative anaylysis of pursuadable with overall customers:
* **More valuable customers:** 'history' 536 vs 242 (around 2.2 times higher), so the target customers already spend more money.
* **Slightly more recent:** 'recency' 5.33 vs 5.76, so the target customers are slightly recent buyers.
* **Strong new 'customer signal':** 'newbie' 77.6% vs 50.2%, so new customers are much more persuadable.
* **Multichannel interaction overrepresented:** 32% vs 12% (around 2.6 times higher). These customers are more engaged and more responsive to campaigns.
* **Phone channel strongly underrepresented:** 21.9% vs 43.8%. Phone-heavy customers are less responsive to E-mail campaigns.
* **Women’s category strongly overrepresented:** 70.7% vs 54.9%. Women-related buyers respond much better to email campaigns.
* **Slight shift toward rural:** 18.8% vs 14.9%. The geographical signal is very week and is not a major decision driver.

Feature importance analysis confirms the above profile with the following top drivers:
* history (strongest)
* recency
* newbie
* channel

## **Final conclusion about this marketing campaign.**
Marketing efforts in similar campaigns should prioritize recently acquired, high-value customers who engage across multiple channels, particularly those with interest in women’s products. Campaigns targeting these segments are expected to yield the highest incremental conversion rates. Conversely, customers primarily interacting via phone channels should be deprioritized, as they show lower responsiveness to email campaigns.